<img src="logo.png" alt="Vegeta" width="240">

# Water propeller — performance in water, CFD check, blade structure, noise and cavitation

The 60 mm three-blade propeller of the survey boat (notebook 16), on its water-cooled pod motor. Water
changes everything about a propeller: 800× the density of air (loads), a free surface a hand's width
above the blades (cavitation), and sound that travels far. The notebook:

```
Boreas in water: thrust/torque/power vs rpm and speed, Ct/Cp/η vs J, cavitation number vs rpm
CAD (Dedalus): the blade as a solid ─► STL/STEP
CFD (Aeromant rotor_mrf, OpenFOAM): thrust and torque at the cruise point, against blade element theory
FEA (Talos): one blade at full-throttle load — stress, deflection, modes vs blade-pass frequency
noise (Boreas): Gutin blade-passing tones and a broadband allowance in water, cavitation inception
export: one JSON for the boat's mission and life work
```

Every coefficient is an explicit input (section polar, cavitation Cp_min, material). The CFD cell
runs OpenFOAM when you run it (`VEGETA_SKIP_OPENFOAM=1` skips it in headless execution).

In [ ]:
import json, math, os, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from vegeta import boreas, dedalus, talos, aeromant
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.dedalus.examples import Propeller as PropellerCAD

RUNS = Path("_runs/water_propeller"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)
WATER = boreas.SEA_WATER                       # rho 1025 kg/m^3, c 1500 m/s
NU_W, G = 1.05e-6, 9.81
# ---- hand-copied from notebook 16 (boat): keep in sync by hand, on purpose ----------------------
V_CRUISE, V_MAX = 1.5, 4.4                     # m/s
WAKE = 0.9                                     # water reaches the propeller at 0.9 x boat speed
R_CRUISE = 3.0                                 # N, resistance at cruise (both are boat-level estimates)
DEPTH_M = 0.15                                 # propeller axis below the surface

## 1. The propeller, the motor, the water

In [ ]:
prop = boreas.Propeller.from_pitch("60 mm 3-blade marine", 0.060, 0.050, blades=3, chord_root_m=0.010, chord_max_m=0.018,
                                   chord_tip_m=0.008, mass_kg=0.020, rotor_mass_kg=0.050, notes="generic planform; fit to the real propeller")
section = boreas.Airfoil(name="marine blade section", cl_alpha=5.5, alpha0_deg=-2.0, cl_max=1.0, cd0=0.02, k=0.05, source="assumed")
CP_MIN = -1.0                                  # section minimum pressure coefficient (assumed; a section property)
motor = boreas.Motor("2836-500KV (pod)", kv_rpm_per_volt=500, resistance_ohm=0.08, no_load_current_a=0.8, max_current_a=30, mass_kg=0.12)
battery = boreas.Battery("4S 10 Ah", cells=4, capacity_ah=10.0, usable_fraction=0.8, mass_kg=0.9)
drive = boreas.Propulsion(prop, section, motor, battery, rho=WATER.density)
cruise = drive.for_thrust(R_CRUISE, WAKE * V_CRUISE)
full = drive.at_throttle(1.0, WAKE * V_MAX)
bollard = drive.at_throttle(1.0, 0.0)          # tied to the quay, full throttle: the highest blade load
pts = {"cruise": cruise, "full speed": full, "bollard pull": bollard}
pd.DataFrame({k: {"rpm": v.rpm, "thrust_N": v.thrust, "torque_Nm": v.aero.torque, "shaft_W": v.aero.power, "current_A": v.current,
                  "prop_eff": v.aero.efficiency, "tip_speed_m_s": v.rpm * 2 * math.pi / 60 * prop.radius, "current_limited": v.current_limited}
              for k, v in pts.items()}).round(3)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for rpm in (1500, 2500, 3500):
    Vs = np.linspace(0.2, 6, 25)
    ops = [boreas.solve(prop, section, rpm, v, WATER.density) for v in Vs]
    J = [o.advance_ratio for o in ops]
    ax[0].plot(J, [o.efficiency for o in ops], label=f"{rpm} rpm"); ax[1].plot(J, [o.ct for o in ops], label=f"{rpm} rpm")
ax[0].set(xlabel="advance ratio J", ylabel="efficiency", ylim=(0, 1), title="η(J) in water"); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].set(xlabel="advance ratio J", ylabel="Ct", title="thrust coefficient"); ax[1].legend(); ax[1].grid(alpha=0.3)
rpms = np.linspace(500, 4500, 21)
for v in (0.0, 1.35, 4.0):
    ax[2].plot(rpms, [boreas.solve(prop, section, r, v, WATER.density).thrust for r in rpms], label=f"{v} m/s inflow")
ax[2].set(xlabel="rpm", ylabel="thrust [N]", title="thrust vs rpm"); ax[2].legend(); ax[2].grid(alpha=0.3)
fig.tight_layout()

### Cavitation: the water-only limit

The cavitation number at 0.7 R, σ = (p_static − p_vapour) / (½ ρ V_rel²), falls with rpm²; when it
drops below −Cp_min of the section, the suction side boils. Inception sets the rpm ceiling more firmly
than the motor does.

In [ ]:
cav = pd.DataFrame([{**boreas.cavitation(prop, r, WAKE * V_CRUISE, DEPTH_M, WATER, cp_min=CP_MIN), "rpm": r} for r in rpms]).set_index("rpm")
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(cav.index, cav["cavitation_number"], label="σ at 0.7 R"); ax.axhline(-CP_MIN, color="#c62828", ls="--", label=f"-Cp_min = {-CP_MIN}")
for name, pt in pts.items():
    ax.axvline(pt.rpm, color="#888", ls=":"); ax.text(pt.rpm, 1.5, name, rotation=90, va="bottom", fontsize=8)
ax.set(xlabel="rpm", ylabel="cavitation number", yscale="log", title=f"cavitation at {DEPTH_M} m depth"); ax.legend(); ax.grid(alpha=0.3, which="both")
incept = boreas.cavitation(prop, full.rpm, WAKE * V_MAX, DEPTH_M, WATER, cp_min=CP_MIN)
print(f"full speed: σ {incept['cavitation_number']:.2f} vs -Cp_min {-CP_MIN}: {'CAVITATES' if incept['cavitates'] else 'no cavitation'}"
      + (f"; inception at ~{incept['rpm_at_inception']:.0f} rpm" if incept['rpm_at_inception'] else ""))

## 2. The blade as a solid (Dedalus)

In [ ]:
CAD_KW = dict(diameter=60.0, pitch=50.0, blades=3, hub_diameter=12.0, hub_height=8.0, bore=4.0, chord_root=10.0, chord_max=18.0,
              chord_tip=8.0, thickness=0.12, camber=0.05, stations=8)
cad = PropellerCAD().generate(**CAD_KW)
files = cad.export(RUNS / "cad", formats=("step", "stl"), stl_tolerance=0.02)
print(f"CAD volume {cad.volume:.0f} mm^3 -> {cad.volume * 1.1e-3:.1f} g in PA12-CF")
dviz.show(dviz.plot3d(cad))
fig = dviz.plot_sections(cad, normal="x", positions=[12.0, 20.0, 27.0], cols=3)

## 3. CFD check of the cruise point (OpenFOAM, rotating frame in water)

`rotor_mrf` with the cruise inflow and rpm, ρ and ν of sea water. Coarse (≈100 k cells, minutes on a
core): expect tens of percent against blade element theory; a negative thrust would mean the handedness
is against `rotation`. The CAD axis (Z) is rotated to +x first.

In [ ]:
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
prop_x = dedalus.Geometry.from_cadquery(cad.shape.rotate((0, 0, 0), (0, 1, 0), 90), name="prop_axis_x")
stl_x = prop_x.export_stl(RUNS / "cad" / "prop_axis_x.stl", tolerance=0.02)
CFD_PARAMS = dict(rpm=cruise.rpm, airspeed=WAKE * V_CRUISE, diameter=0.060, kinematic_viscosity=NU_W, density=WATER.density, rotation=1,
                  iterations=400, cells_per_diameter=6.0, surface_level=3, near_level=2, rotor_level=2, wake_level=1)
cfd = None
if RUN_CFD:
    case = aeromant.CFDCase("rotor_mrf", stl_x, CFD_PARAMS, workdir=RUNS / "cfd_cruise", geometry_units="mm",
                            environment=aeromant.OpenFOAMEnvironment.detect())
    print(case.prepare(overwrite=True))
    cfd = case.run(progress=True)
    print(cfd)
else:
    print("CFD skipped (VEGETA_SKIP_OPENFOAM=1): run this cell on a machine with OpenFOAM")

In [ ]:
if cfd is not None and cfd.ok:
    m = cfd.metrics
    compare = pd.DataFrame({"BEMT (Boreas)": {"thrust_N": cruise.thrust, "torque_Nm": cruise.aero.torque, "power_W": cruise.aero.power, "efficiency": cruise.aero.efficiency},
                            "CFD (rotor_mrf)": {"thrust_N": m["thrust_N"], "torque_Nm": m["torque_Nm"], "power_W": m["power_W"], "efficiency": m["efficiency"]}})
    compare["CFD / BEMT"] = compare["CFD (rotor_mrf)"] / compare["BEMT (Boreas)"]
    display(compare.round(3))
    aviz.show(aviz.plot_field_slice(case, "U", normal="z"))
    fig = aviz.plot_section(case, "p", normal="z", zoom=2)      # the suction-side pressure minimum is the cavitation indicator
    aviz.show(aviz.plot_surface_pressure(case))

## 4. Blade structure (Talos): one blade at bollard pull, and its modes

The single blade with its hub is meshed; the hub faces are fixed; the blade's share of the thrust and
of the torque (at 0.7 R) is applied as tractions over the blade surface — the right totals, an
approximate distribution. Printed PA12-CF (E = 3.5 GPa, yield 60 MPa) — a bronze or stainless blade
would be the usual choice; the numbers say whether the printed one is far from the limit.

Modes are computed in air; in water the added mass lowers them (roughly ×0.6–0.7 for a thin blade),
which is applied as a factor before comparing with the blade-passing frequency.

In [ ]:
blade = PropellerCAD().generate(**dict(CAD_KW, blades=1))
bfiles = blade.export(RUNS / "cad_blade", formats=("step",))
hub_r, hub_h, R = CAD_KW["hub_diameter"] / 2, CAD_KW["hub_height"], CAD_KW["diameter"] / 2
REGIONS = [talos.SurfacesInBox("hub", (-hub_r - 0.5, -hub_r - 0.5, -hub_h / 2 - 0.5, hub_r + 0.5, hub_r + 0.5, hub_h / 2 + 0.5)),
           talos.SurfacesInBox("blade", (hub_r - 1.5, -25.0, -25.0, R + 1.0, 25.0, 25.0))]
PA12CF = talos.Material("PA12-CF (printed)", youngs_modulus=3500.0, poissons_ratio=0.40, density=1.1e-9, yield_strength=60.0, source="nominal, flat orientation")
T_blade = bollard.thrust / prop.blades                          # N per blade at bollard pull
F_tan = bollard.aero.torque / (prop.blades * 0.7 * prop.radius)  # tangential force per blade at 0.7 R
# the blade points along +x; thrust acts along the axis (z); the tangential force along y
model = talos.StructuralModel(bfiles.artifacts["step"], "mm-N-MPa", PA12CF, REGIONS, [talos.FixedSupport("hub")],
                              [talos.Force("blade", fz=T_blade, fy=-F_tan)], talos.MeshSettings(element_size=1.0), name="bollard")
mesh_res = model.mesh(RUNS / "blade_fea", progress=True)
res = model.solve(RUNS / "blade_fea", progress=True)
print(f"per blade: thrust {T_blade:.2f} N, tangential {F_tan:.2f} N at bollard pull ({bollard.rpm:.0f} rpm)")
print(res)
tviz.show(tviz.plot_problem(model, mesh_res, arrow_scale=8))

In [ ]:
if res.ok:
    tviz.show(tviz.plot_results(res, field="von_mises"))
    fig = tviz.plot_section(res, normal="x", origin=(10.0, 0, 0), field="von_mises")      # root section

In [ ]:
shutil.copytree(RUNS / "blade_fea", RUNS / "blade_modal", dirs_exist_ok=True)
modes = model.solve_modes(RUNS / "blade_modal", n_modes=4)      # same mesh and supports; the loads play no part in a modal solve
ADDED_MASS_FACTOR = 0.65                                       # in-water frequency / in-air frequency, assumed for a thin blade
f_air = modes.metrics["frequencies_hz"]
f_water = [f * ADDED_MASS_FACTOR for f in f_air]
bpf = {k: boreas.excitations(prop, v.rpm)["blade_pass_hz"] for k, v in pts.items()}
print("blade modes in air:", [round(f) for f in f_air], "Hz; in water (x0.65):", [round(f) for f in f_water], "Hz")
print("blade-pass frequencies:", {k: round(v, 1) for k, v in bpf.items()}, "Hz -> margin to the first wet mode:",
      {k: f"{abs(f_water[0] - v) / f_water[0]:.0%}" for k, v in bpf.items()})
tviz.show(tviz.plot_mode(modes, mode=1))

## 5. Noise: blade-passing tones and cavitation

Gutin's steady-loading tones at 1 m, broadside (90° from the axis), in dB re 1 µPa — the underwater
convention. A broadband allowance is added in power. Above cavitation inception the noise is dominated
by collapsing bubbles, which no loading model predicts: the cavitation margin is the real design number.

In [ ]:
DIST = 1.0
rows = {}
for name, pt in pts.items():
    tones = boreas.gutin_harmonics(prop, pt.thrust, pt.aero.torque, pt.rpm, DIST, 90.0, WATER, harmonics=5)
    bb = boreas.broadband_level(prop, pt.thrust, pt.rpm, DIST, WATER)
    total = 10 * math.log10(10 ** (tones["total_tonal_db"] / 10) + 10 ** (bb / 10))
    c = boreas.cavitation(prop, pt.rpm, WAKE * V_CRUISE if name == "cruise" else (WAKE * V_MAX if name == "full speed" else 0.0), DEPTH_M, WATER, cp_min=CP_MIN)
    rows[name] = {"rpm": pt.rpm, "BPF_hz": tones["blade_pass_hz"], "tonal_dB_re_1uPa_1m": tones["total_tonal_db"], "broadband_dB": bb,
                  "total_dB": total, "cavitation_number": c["cavitation_number"], "cavitates": c["cavitates"]}
    if name == "cruise":
        fig, ax = plt.subplots(figsize=(6.5, 3.4))
        ax.bar(tones["frequency_hz"], tones["spl_db"], width=8, label="Gutin tones (cruise)")
        ax.axhline(bb, color="#c62828", ls="--", label="broadband allowance"); ax.set(xlabel="frequency [Hz]", ylabel="dB re 1 µPa at 1 m"); ax.legend(); ax.grid(alpha=0.3)
pd.DataFrame(rows).T.round(1)

## 6. Export

In [ ]:
doc = {"propeller": prop.describe(), "section": section.__dict__, "medium": WATER.__dict__, "cp_min": CP_MIN, "depth_m": DEPTH_M,
       "points": {k: v.to_dict() for k, v in pts.items()}, "cavitation_full_speed": incept,
       "blade_fea_bollard": {"thrust_per_blade_N": T_blade, "tangential_per_blade_N": F_tan, "metrics": {k: v for k, v in res.metrics.items() if not isinstance(v, (dict, list))}},
       "blade_modes_hz_air": f_air, "blade_modes_hz_water_assumed": f_water, "added_mass_factor": ADDED_MASS_FACTOR,
       "noise": rows, "cfd_cruise": ({"parameters": CFD_PARAMS, "metrics": {k: v for k, v in cfd.metrics.items() if not isinstance(v, (list, dict))}} if cfd is not None and cfd.ok else "not run")}
(RUNS / "water_propeller.json").write_text(json.dumps(doc, indent=2, default=float))
print("written:", sorted(x.name for x in RUNS.iterdir()))

**Reading the result:** cavitation inception versus the operating rpm is the limit that matters; the
CFD check (run it) says how far the coarse blade element numbers are off for this planform; the blade
stress at bollard pull tells whether a printed blade is acceptable or the propeller must be metal; and
the tones say what a hydrophone would hear before cavitation. All of it is recorded with its assumptions.